# Day 34 — Monitoring, drift & the retraining loop

The RAG service is live (Days 32–33). Production isn't a state you reach — it's a **stream**
you watch. Inputs shift, the corpus changes, a provider degrades, and quality erodes without a
single error in the logs.

Today, hands-on: the **golden signals** for an LLM app, **distribution drift** detection on
the inputs (PSI + KS from scratch), and the **refresh loop** that closes back to Day 33 —
detect decay → collect + label → update → eval-gate → canary → promote.

This is the last lesson. It ties Weeks 5–10 into one operating system.

## Learning objectives

1. Name the golden signals for an LLM service and build an aggregator over a trace stream.
2. Compute Population Stability Index and a KS test to detect input drift; set thresholds.
3. Distinguish data drift, concept drift, and quality decay — and which signal catches each.
4. Implement a `RefreshTrigger` that combines drift + eval-score decay + user feedback.
5. Run a full simulated refresh loop and show the eval gate + canary from Day 33 close it.
6. List the feedback-loop hazards: training on your own outputs, feedback bias, metric gaming.

## Agenda (60 min)

| # | Segment | Time |
| - | ------- | ---- |
| 0 | Production is a stream, not a state | 3 min |
| 1 | Monitoring only latency, and missing the collapse | 7 min |
| 2 | Golden signals: an aggregator over the trace stream | 13 min |
| 3 | Drift from scratch: PSI and KS | 14 min |
| 4 | Alerting, sampling, PII — the operational layer | 6 min |
| 5 | The refresh loop, end to end | 14 min |
| 6 | Feedback hazards; course wrap-up | 3 min |
| 7 | Exercises and self-check quiz | — |

## Setup

```bash
source ../../../.venv/bin/activate
```

Uses `numpy` and `scipy` (already installed). No API calls — the trace stream is synthetic.


## 0 — Production is a stream, not a state (3 min)

You shipped a system that scored 0.83 on the eval set. That number describes **one moment
against one frozen dataset**. In production:

- the *questions* users ask drift (new product launched, incident trending) — **data drift**
- the *right answer* changes (policy updated, price changed) even for the same question —
  **concept drift**
- the *corpus* you retrieve from goes stale or grows noisy — **retrieval decay**
- a provider changes a model behind the same name, or degrades — **silent model drift**

None of these throw an exception. Monitoring is how you see them; the refresh loop is how you
respond.

## 1 — Monitoring only latency, and missing the collapse (7 min)

In [1]:
import numpy as np
rng = np.random.default_rng(0)

def make_stream(n, *, corpus_version="v7", topic_mix=(0.7, 0.3)):
    """Each event: a served RAG request. 'retrieval_hit' depends on whether the corpus
       still covers the question's topic."""
    events = []
    for _ in range(n):
        topic = rng.choice(["billing", "new_product"], p=topic_mix)
        # v7 corpus has no 'new_product' docs -> retrieval misses, answer ungrounded
        covered = not (topic == "new_product" and corpus_version == "v7")
        hit = covered and rng.random() < 0.92
        grounded = hit and rng.random() < 0.95
        events.append(dict(
            topic=topic,
            latency_ms=float(rng.normal(1100, 250)),
            error=rng.random() < 0.01,
            retrieval_hit=hit,
            grounded=grounded,
            tokens_in=int(rng.normal(1200, 200)), tokens_out=int(rng.normal(180, 40)),
            thumbs_down=(not grounded) and rng.random() < 0.4,
        ))
    return events

week1 = make_stream(2000, topic_mix=(0.7, 0.3))
week4 = make_stream(2000, topic_mix=(0.35, 0.65))   # 'new_product' questions took over

def latency_only(events):
    lat = np.array([e["latency_ms"] for e in events])
    return dict(p50=round(np.percentile(lat, 50)), p95=round(np.percentile(lat, 95)),
               error_rate=round(np.mean([e["error"] for e in events]), 3))

print("week 1:", latency_only(week1))
print("week 4:", latency_only(week4))
print("\nlatency + errors look identical. now the signals that matter:")
for label, ev in [("week 1", week1), ("week 4", week4)]:
    print(f"  {label}: retrieval_hit={np.mean([e['retrieval_hit'] for e in ev]):.2f}  "
          f"grounded={np.mean([e['grounded'] for e in ev]):.2f}  "
          f"thumbs_down={np.mean([e['thumbs_down'] for e in ev]):.2f}")


week 1: {'p50': 1097, 'p95': 1511, 'error_rate': np.float64(0.008)}
week 4: {'p50': 1106, 'p95': 1510, 'error_rate': np.float64(0.008)}

latency + errors look identical. now the signals that matter:
  week 1: retrieval_hit=0.63  grounded=0.60  thumbs_down=0.16
  week 4: retrieval_hit=0.31  grounded=0.30  thumbs_down=0.27


Latency p50/p95 and error rate are **flat**. Meanwhile grounded-answer rate fell from ~0.85
to ~0.55 because two-thirds of traffic now asks about a product the `v7` corpus never
ingested. An ops dashboard watching infra metrics sees nothing.

## 2 — Golden signals: an aggregator over the trace stream (13 min)

Every request emits a trace (Day 27). Aggregate it into a fixed signal set on a rolling
window. For an LLM app the signals are:

| Family | Signal | Why |
| ------ | ------ | --- |
| Latency | p50 / p95 / p99 | UX + timeout budget |
| Traffic | req/s, tokens/s | capacity, cost driver |
| Errors | error rate, refusal rate, timeout rate | reliability |
| Cost | $/req, $/day | the bill (Day 29) |
| **Quality** | retrieval hit rate, grounded rate, judge score on a sample | the thing users feel |
| **Feedback** | thumbs-down rate, escalation rate | ground truth, lagging |

In [2]:
PRICE = {"in": 0.80 / 1e6, "out": 4.00 / 1e6}   # $/token, claude-haiku-4-5

class SignalAggregator:
    def __init__(self, window=500):
        self.window = window
        self.buf = []

    def observe(self, event):
        self.buf.append(event)
        if len(self.buf) > self.window:
            self.buf.pop(0)

    def signals(self):
        e = self.buf
        n = len(e)
        lat = np.array([x["latency_ms"] for x in e])
        cost = np.array([x["tokens_in"] * PRICE["in"] + x["tokens_out"] * PRICE["out"] for x in e])
        return {
            "n": n,
            "p50_ms": round(float(np.percentile(lat, 50))),
            "p95_ms": round(float(np.percentile(lat, 95))),
            "p99_ms": round(float(np.percentile(lat, 99))),
            "error_rate": round(float(np.mean([x["error"] for x in e])), 3),
            "retrieval_hit_rate": round(float(np.mean([x["retrieval_hit"] for x in e])), 3),
            "grounded_rate": round(float(np.mean([x["grounded"] for x in e])), 3),
            "thumbs_down_rate": round(float(np.mean([x["thumbs_down"] for x in e])), 3),
            "cost_per_req_usd": round(float(np.mean(cost)), 5),
        }

agg = SignalAggregator(window=500)
for ev in week1[-500:]:
    agg.observe(ev)
print("baseline signals:", agg.signals())

# feed week 4 and watch quality signals move while infra signals don't
agg2 = SignalAggregator(window=500)
for ev in week4[-500:]:
    agg2.observe(ev)
print("week-4 signals :", agg2.signals())


baseline signals: {'n': 500, 'p50_ms': 1090, 'p95_ms': 1521, 'p99_ms': 1713, 'error_rate': 0.016, 'retrieval_hit_rate': 0.6, 'grounded_rate': 0.566, 'thumbs_down_rate': 0.19, 'cost_per_req_usd': 0.00168}
week-4 signals : {'n': 500, 'p50_ms': 1102, 'p95_ms': 1524, 'p99_ms': 1663, 'error_rate': 0.012, 'retrieval_hit_rate': 0.308, 'grounded_rate': 0.298, 'thumbs_down_rate': 0.278, 'cost_per_req_usd': 0.00168}


You'd export these to Prometheus/CloudWatch and alert on the quality signals with the same
seriousness as on error rate. A drop in `grounded_rate` is a page, not a nice-to-know.

## 3 — Drift from scratch: PSI and KS (14 min)

Quality signals tell you *something is wrong now*. Drift detection on the **inputs** can warn
you *before* quality drops, and points at *what* changed.

**Population Stability Index** — bin a reference sample and a current sample the same way;
sum `(cur% − ref%) · ln(cur% / ref%)` over bins. Rule of thumb: `<0.1` stable, `0.1–0.25`
moderate, `>0.25` significant.

In [3]:
def psi(reference, current, bins=10):
    ref = np.asarray(reference, float)
    cur = np.asarray(current, float)
    edges = np.quantile(ref, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    ref_pct = np.histogram(ref, edges)[0] / len(ref)
    cur_pct = np.histogram(cur, edges)[0] / len(cur)
    eps = 1e-6
    ref_pct = np.clip(ref_pct, eps, None)
    cur_pct = np.clip(cur_pct, eps, None)
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))

# 1-D proxy for "embedding of the incoming question" — e.g. projection on the top PC.
ref_emb  = rng.normal(0.0, 1.0, 3000)
same_emb = rng.normal(0.0, 1.0, 1500)
shift_emb = rng.normal(0.7, 1.2, 1500)     # topic mix moved

print(f"PSI (no shift)   : {psi(ref_emb, same_emb):.3f}")
print(f"PSI (shifted)    : {psi(ref_emb, shift_emb):.3f}")


PSI (no shift)   : 0.015
PSI (shifted)    : 0.352


In [4]:
from scipy.stats import ks_2samp

def ks_drift(reference, current, alpha=0.01):
    stat, p = ks_2samp(reference, current)
    return {"ks_stat": round(float(stat), 3), "p_value": float(f"{p:.2e}"),
            "drift": bool(p < alpha)}

print("KS (no shift):", ks_drift(ref_emb, same_emb))
print("KS (shifted) :", ks_drift(ref_emb, shift_emb))

# categorical drift (topic label) — PSI works directly on the category proportions
def categorical_psi(ref_labels, cur_labels):
    cats = sorted(set(ref_labels) | set(cur_labels))
    ref = np.array([np.mean([l == c for l in ref_labels]) for c in cats])
    cur = np.array([np.mean([l == c for l in cur_labels]) for c in cats])
    eps = 1e-6; ref = np.clip(ref, eps, None); cur = np.clip(cur, eps, None)
    return float(np.sum((cur - ref) * np.log(cur / ref)))

t1 = [e["topic"] for e in week1]
t4 = [e["topic"] for e in week4]
print(f"\ncategorical PSI on topic label: {categorical_psi(t1, t4):.3f}  (>0.25 -> significant)")


KS (no shift): {'ks_stat': 0.035, 'p_value': 0.171, 'drift': False}
KS (shifted) : {'ks_stat': 0.241, 'p_value': 2.1e-51, 'drift': True}

categorical PSI on topic label: 0.463  (>0.25 -> significant)


| Drift type | What moved | Best detector |
| ---------- | ---------- | ------------- |
| **Data / covariate** | the input distribution (topics, phrasing, length) | PSI / KS on input embeddings & metadata |
| **Concept** | P(answer \| input) — same question, new correct answer | eval score on a *fresh-labelled* set; feedback |
| **Retrieval decay** | corpus coverage / freshness | retrieval hit rate; PSI on retrieved-doc age |
| **Model drift** | provider changed the model | canary the same eval set over time |

PSI/KS can't see concept drift — the inputs look the same. Only re-labelled evals and user
feedback catch that.

## 4 — Alerting, sampling, PII — the operational layer (6 min)

- **Alert on symptoms, page on impact.** `grounded_rate < 0.7` for 15 min → page. `PSI > 0.25`
  → ticket (investigate, not necessarily 2am). Multi-window burn-rate alerts beat single
  thresholds (fewer false pages).
- **Sample the traces you keep.** 100% of metrics, but store full prompt/response for ~1–5%
  (plus 100% of errors and thumbs-down). LLM traces are large and sensitive.
- **Scrub before you store.** Run prompts/responses through a PII redactor before they hit
  logs. Never log API keys or raw auth headers. Set a retention window (e.g. 30 days) and
  honour deletion requests.
- **Keep an eval "canary set"** — run the frozen Week 9 suite against production daily; a
  drop with no deploy = the provider changed something.

In [5]:
import re
def redact(text):
    text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.-]+", "<email>", text)
    text = re.sub(r"\b(?:\d[ -]*?){13,16}\b", "<card>", text)
    text = re.sub(r"sk-[A-Za-z0-9_-]{12,}", "<secret>", text)
    return text

print(redact("Contact jane.doe@acme.com, card 4111 1111 1111 1111, key sk-ant-abc123def456ghi789"))


Contact <email>, card <card>, key <secret>


## 5 — The refresh loop, end to end (14 min)

Close the loop back to Day 33. A `RefreshTrigger` fuses three signals into a decision; when it
fires, the pipeline collects recent good cases, updates the system (here: re-index + refresh
few-shot examples), runs the **eval gate**, and if it passes, hands off to the **canary**.

In [6]:
from dataclasses import dataclass

@dataclass
class RefreshTrigger:
    psi_limit: float = 0.25
    grounded_floor: float = 0.75
    eval_decay: float = 0.05          # drop vs shipped baseline
    thumbs_down_ceiling: float = 0.10

    def decide(self, *, psi_value, grounded_rate, eval_score, baseline_eval, thumbs_down_rate):
        reasons = []
        if psi_value > self.psi_limit:
            reasons.append(f"input drift PSI={psi_value:.2f}")
        if grounded_rate < self.grounded_floor:
            reasons.append(f"grounded_rate={grounded_rate:.2f}")
        if baseline_eval - eval_score > self.eval_decay:
            reasons.append(f"eval decay {baseline_eval:.2f}->{eval_score:.2f}")
        if thumbs_down_rate > self.thumbs_down_ceiling:
            reasons.append(f"thumbs_down={thumbs_down_rate:.2f}")
        return {"refresh": len(reasons) >= 2, "reasons": reasons}   # need 2+ to avoid twitchiness

trig = RefreshTrigger()
now = {"psi_value": categorical_psi(t1, t4),
       "grounded_rate": agg2.signals()["grounded_rate"],
       "eval_score": 0.71, "baseline_eval": 0.83,
       "thumbs_down_rate": agg2.signals()["thumbs_down_rate"]}
print("trigger:", trig.decide(**now))


trigger: {'refresh': True, 'reasons': ['input drift PSI=0.46', 'grounded_rate=0.30', 'eval decay 0.83->0.71', 'thumbs_down=0.28']}


In [7]:
# the loop: collect -> update -> eval-gate -> canary -> promote
def collect_good_cases(events, k=6):
    """Mine recent grounded, thumbs-up answers as new few-shot exemplars / eval cases."""
    good = [e for e in events if e["grounded"] and not e["thumbs_down"]]
    return good[-k:]

def rebuild_system(corpus_version, new_examples):
    # re-ingest so 'new_product' is covered now, refresh few-shots
    return {"corpus_version": "v8", "few_shot_n": len(new_examples)}

def run_eval(system):
    # v8 corpus covers the new topic -> grounded rate (and score) recover
    return 0.86 if system["corpus_version"] == "v8" else 0.71

def canary_ok(system):
    return run_eval(system) >= 0.83     # stands in for the Day 33 Canary().roll_out()

def refresh_loop(events, trigger_decision, baseline_eval=0.83):
    if not trigger_decision["refresh"]:
        return "no action"
    print("  refresh fired:", "; ".join(trigger_decision["reasons"]))
    examples = collect_good_cases(events)
    print(f"  collected {len(examples)} recent good cases -> new eval + few-shot set")
    candidate = rebuild_system("v7", examples)
    score = run_eval(candidate)
    print(f"  candidate eval: {score:.2f} vs baseline {baseline_eval:.2f}")
    if score - baseline_eval < -0.03:
        return "BLOCKED at eval gate"
    if not canary_ok(candidate):
        return "ROLLED BACK at canary"
    return f"PROMOTED corpus {candidate['corpus_version']} (eval {score:.2f})"

print(refresh_loop(week4, trig.decide(**now)))


  refresh fired: input drift PSI=0.46; grounded_rate=0.30; eval decay 0.83->0.71; thumbs_down=0.28
  collected 6 recent good cases -> new eval + few-shot set
  candidate eval: 0.86 vs baseline 0.83
PROMOTED corpus v8 (eval 0.86)


That is the whole operating loop:

```
serve ──► trace ──► signals ──► drift / decay / feedback
                                      │
                          RefreshTrigger (2+ reasons)
                                      │
     collect good cases ──► rebuild (re-index / re-tune / new few-shots)
                                      │
                          eval gate (Day 33) ──► canary (Day 33) ──► promote
                                      │
                                   new baseline ──► back to serve
```

## 6 — Feedback hazards; course wrap-up (3 min)

The loop can quietly poison itself:

- **Training on your own outputs.** If new few-shots / fine-tune data are the model's past
  answers (lightly filtered), errors compound and diversity collapses. Prefer human-labelled
  or human-corrected cases; if you must use model outputs, gate them through a judge *and* a
  sample of human review.
- **Feedback bias.** Thumbs-down is given ~5× more than thumbs-up, and only by certain users
  on certain topics. Don't treat the feedback stream as a representative label set.
- **Metric gaming (Goodhart, Day 10).** Optimising `grounded_rate` can teach the model to
  always cite *something*, relevant or not. Keep a held-out human eval the loop can't touch.
- **Drift-chasing.** Not every PSI spike needs a refresh — a one-day news event isn't concept
  drift. The "2+ reasons" rule and a cooldown period matter.

### Course wrap-up

Ten and a bit weeks, from `text → numbers` to a self-refreshing production system:

| Weeks | You can now |
| ----- | ----------- |
| 1–2 | explain tokenisation, attention, context windows, and prompt like an engineer |
| 3–4 | choose fine-tune vs RAG vs prompt and defend it with evals, not vibes |
| 5–6 | build embeddings + a vector index + a grounded RAG pipeline with citations |
| 7–8 | build a tool-using agent and drive the Anthropic API (tools, streaming, MCP) |
| 9 | write evals, an eval harness, and tracing that a CI gate can use |
| 10–11 | serve it, cost it, containerise it, declare its infra, ship it through an eval-gated canary, and keep it healthy with drift detection and a refresh loop |

Next: the **[capstone](../../practice/capstone/)** — build the whole thing as one system.

**Where this goes next:** you're done with the daily lessons. Do the capstone, then pick a
real problem at work and run this loop on it.

## Exercises

1. **p99 alerting.** Add a multi-window burn-rate check to `SignalAggregator`: page only if
   `grounded_rate` is below floor in *both* a 5-min and a 1-hour window. Simulate a 2-minute
   blip and a 90-minute outage; show only the second pages.
2. **PSI bin sensitivity.** Recompute the shifted-embedding PSI for `bins ∈ {5, 10, 20, 50}`.
   How stable is the verdict? What breaks with too many bins on a small current sample?
3. **Concept drift, not data drift.** Construct a stream where the *inputs* are unchanged
   (PSI ≈ 0) but the correct answers changed (a policy update). Show PSI/KS miss it and a
   re-labelled eval catches it.
4. **Retrieval-age drift.** Add a `retrieved_doc_age_days` field. Detect corpus staleness with
   PSI on that distribution and wire it as a fifth `RefreshTrigger` reason.
5. **Trigger tuning.** Make `RefreshTrigger` require reasons to persist for N consecutive
   windows (a cooldown), and add a minimum interval between refreshes. Re-run §5 with a
   one-day news spike and show it does *not* fire.
6. **Poisoned loop.** Run `refresh_loop` for 5 generations where `collect_good_cases` returns
   the model's *own* prior answers with no human check and a 10% error slips through each
   time. Track eval score across generations. What do you observe, and what one change fixes it?

## Self-check quiz

1. Give one drift type that latency + error-rate monitoring will never catch, and the signal
   that does.
2. What does PSI measure, and what are the rough thresholds for "moderate" and "significant"?
3. Why can't PSI or a KS test detect concept drift?
4. Name the six golden-signal families for an LLM service. Which two do infra dashboards
   usually miss?
5. Why does `RefreshTrigger` require 2+ reasons instead of firing on any one?
6. What is the risk of building new training / few-shot data from the model's own production
   outputs, and how do you mitigate it?
7. Sketch the refresh loop from "serve" back to "serve", naming the Day 33 stages it reuses.
